In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:512'
import sys
import glob
import torch
import torch.nn.functional as F
import pandas as pd
from PIL import Image
from torchvision import transforms
try:
    import ipywidgets
    from tqdm.notebook import tqdm
    print('tqdm mode: notebook')
except ImportError:
    from tqdm import tqdm
    print('tqdm mode: text (ipywidgets tidak tersedia)')

# Tambahkan path ke arsitektur model
sys.path.append(os.path.abspath("../Model_Architecture"))
from model_architecture_lengkap import SampahClassifier, dapatkan_transform_val, UKURAN_INPUT

In [ ]:
ROOT = os.path.abspath("..")
TEST_DIR = os.path.join(ROOT, "test")
MODEL_PATH = os.path.join(ROOT, "Train_model", "Best_model", "best_model_final.pth")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# Label Mapping dari project overview
# 0: Recyclable, 1: Electronic, 2: Organic
IDX_TO_LABEL = {0: "Recyclable", 1: "Electronic", 2: "Organic"}

In [ ]:
print("Memuat arsitektur model...")
model = SampahClassifier(num_classes=3, pretrained=False)
print("Memuat bobot model terbaik...")
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE, weights_only=True))
model.to(DEVICE)
model.eval()
print("Model siap digunakan!")

In [ ]:
transform_val = dapatkan_transform_val(UKURAN_INPUT)

def proses_gambar_test(img_path):
    '''
    Fungsi untuk membaca dan melakukan downsample gambar sebelum transform.
    Sesuai catatan EDA, ada gambar yang sangat besar (bisa mencapai belasan MB).
    '''
    try:
        with Image.open(img_path) as img:
            lebar, tinggi = img.size
            MAX_SISI = 2048 # Sesuai dengan batasan preprocessing sebelumnya
            if max(lebar, tinggi) > MAX_SISI:
                rasio = MAX_SISI / max(lebar, tinggi)
                ukuran_baru = (int(lebar * rasio), int(tinggi * rasio))
                img = img.resize(ukuran_baru, Image.BILINEAR)
            
            img_rgb = img.convert("RGB")
            
        img_tensor = transform_val(img_rgb)
        return img_tensor.unsqueeze(0) # Tambah batch dimension
    except Exception as e:
        print(f"Error membaca gambar {img_path}: {e}")
        # Kembalikan tensor hitam jika gagal
        return torch.zeros(1, 3, UKURAN_INPUT, UKURAN_INPUT)

In [ ]:
# Ambil 200 gambar pertama dari folder test (diurutkan berdasarkan angka jika memungkinkan)
test_files = glob.glob(os.path.join(TEST_DIR, "*.jpg"))
# Sort secara numerik: "test/1.jpg" -> 1
test_files = sorted(test_files, key=lambda x: int(os.path.basename(x).split('.')[0]))
sample_files = test_files[:200]

hasil_prediksi = []

print("Melakukan inferensi pada 200 gambar sample...")
torch.cuda.empty_cache()
with torch.no_grad():
    for fpath in tqdm(sample_files, desc="Inference"):
        filename = os.path.basename(fpath)
        img_tensor = proses_gambar_test(fpath).to(DEVICE)
        
        # Forward pass
        output = model(img_tensor)
        
        # Hitung probabilitas dengan softmax
        probs = F.softmax(output, dim=1)[0]
        prob_recyclable = probs[0].item()
        prob_electronic = probs[1].item()
        prob_organic = probs[2].item()
        
        # Prediksi kelas (argmax)
        pred_idx = torch.argmax(probs).item()
        pred_label = IDX_TO_LABEL[pred_idx]
        
        hasil_prediksi.append({
            "File": filename,
            "Prob_Recyclable": round(prob_recyclable, 4),
            "Prob_Electronic": round(prob_electronic, 4),
            "Prob_Organic": round(prob_organic, 4),
            "Prediksi": pred_label
        })

df_hasil = pd.DataFrame(hasil_prediksi)
print("\nInference selesai!")

In [ ]:
# Tampilkan 20 data teratas untuk preview
display(df_hasil.head(20))

# Simpan ke CSV
output_csv = "prediksi_200_sample.csv"
df_hasil.to_csv(output_csv, index=False)
print(f"Hasil prediksi 200 sample telah disimpan ke {output_csv}")

# Distribusi hasil prediksi dari 200 sample
print("\nDistribusi Prediksi 200 Sample:")
print(df_hasil['Prediksi'].value_counts())